# Higgs Boson Decaying to Tau Leptons

<div>
<img src="https://raw.githubusercontent.com/illinois-ipaml/MachineLearningForPhysics/main/img/Project_HiggsTauTau-EventDisplay.jpg" width=700></img>
</div>

*Event display from the ATLAS experiment showing a candidate Higgs boson decaying to a pair of τ leptons. Image: ATLAS Experiment © CERN.*

## <span style="color:Orange">Overview</span>

In [Project 01](https://illinois-ipaml.github.io/MachineLearningForPhysics/_sources/projects/Project_01_HiggsTauTau.html) you explored simulated $gg\to H\to\tau^+\tau^-$ (signal) and $Z\to\tau^+\tau^-$ (background) events and studied which features separate them. In this project you will train neural-network classifiers on these events and reproduce the main result of Refs. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html) and [[2]](https://doi.org/10.1103/PhysRevLett.114.111801): a **deep** network trained only on the low-level detector measurements can match or beat a **shallow** network that also uses the high-level features physicists constructed by hand.

## <span style="color:Orange">Data Source</span>

The data are the same as in Project 01: **Higgs to Tau Tau dataset** (UCI MLPhysics portal, D. Whiteson) [[3]](https://mlphysics.ics.uci.edu/data/htautau/)

* Signal: https://mlphysics.ics.uci.edu/data/htautau/htautau.txt.gz (44.3 million events, 3.8 GB)
* Background: https://mlphysics.ics.uci.edu/data/htautau/ztautau.txt.gz (42.8 million events, 3.7 GB)

Each file contains whitespace-separated rows of 25 features, with no header and no label column. **Columns 0–9 are the low-level features and columns 10–24 are the high-level features.** Use the column identifications you made in Project 01.

The cell below streams the first `N_EVENTS` events of each file (no full download), adds labels, and splits them into training, validation and test sets. Start small while you develop your code, then increase `N_EVENTS` for your final results. A shallow network on a CPU is comfortable with a few hundred thousand events per class. On a GPU, aim for several million.

In [ ]:
import gzip
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

BASE_URL = "https://mlphysics.ics.uci.edu/data/htautau/"
N_EVENTS = 250_000  # per class; increase for your final results

def read_events(filename, nrows):
    """Stream the first `nrows` events of a gzipped file without downloading all of it."""
    with urllib.request.urlopen(BASE_URL + filename) as response, gzip.GzipFile(fileobj=response) as f:
        return pd.read_csv(f, sep=r"\s+", header=None, nrows=nrows)

sig = read_events("htautau.txt.gz", N_EVENTS)
bkg = read_events("ztautau.txt.gz", N_EVENTS)

X = np.vstack([sig.to_numpy(), bkg.to_numpy()]).astype(np.float32)
y = np.concatenate([np.ones(len(sig)), np.zeros(len(bkg))]).astype(np.float32)

LOW = list(range(0, 10))    # low-level features
HIGH = list(range(10, 25))  # high-level features

# 60% train / 20% validation / 20% test, stratified by class
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = (scaler.transform(a) for a in (X_train, X_val, X_test))
print({name: a.shape for name, a in [("train", X_train), ("val", X_val), ("test", X_test)]})

## <span style="color:Orange">Questions</span>

**General instructions**

* Use PyTorch, as in the *Deep Learning* and *GPU 101* lectures.
* Keep a **test set** that is never used for training, hyperparameter tuning or early stopping. Use a separate **validation set** for those.
* Fit any input scaling (e.g. `StandardScaler`) on the training set only, then apply it to the validation and test sets.
* Plot training and validation loss versus epoch for every network you train, so you can see whether it has converged or overfit.
* For each network, report the number of trainable parameters, the training time, and the hardware used (CPU or GPU model).
* The papers trained on tens of millions of events for hundreds of epochs. You are **not** expected to match their AUC values exactly. Explain any differences you see.

### <span style="color:LightGreen">Question 01</span>

**Shallow network.** Implement and train a shallow neural network (one hidden layer) like those in Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html). Choose its hyperparameters from Table 1 of that paper, or use an even smaller network so that training is manageable on a CPU. Use the complete set of 25 features. Section 3.3 of the paper describes the remaining hyperparameters it used: tanh hidden units, a logistic output, mini-batches of 100, and momentum. You may use a modern optimizer such as Adam instead, but say so.

Show the distribution of the network output for signal and for background test events, overlaid and normalized to unit area (analogous to Fig. 4 of Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html)), the ROC curve, and the AUC on the test set.

### <span style="color:LightGreen">Question 02</span>

**Deep network.** Train a fully-connected deep neural network (DNN) on a GPU, with meaningfully more trainable parameters than your shallow network from Question 01. Table 2 of Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html) gives the depths (3–6 hidden layers) and widths (100 or 300 units) the paper considered. The exact architecture is up to you. Use the complete feature set, and show the same output-distribution plot, ROC curve and AUC as in Question 01. Overlay the shallow and deep ROC curves on one plot.

### <span style="color:LightGreen">Question 03</span>

**Which features matter?** Retrain your deep network (and, if time allows, your shallow network) using (a) only the 10 low-level features, (b) only the 15 high-level features, and (c) all 25 features. Summarize the test-set AUCs in a table like Table 3 of Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html).

* Does the deep network trained on the low-level features alone recover the information in the hand-crafted high-level features?
* How do your conclusions compare with those of the paper?
* What does this imply for how physicists design analyses?

*Optional extensions*

* Convert your best AUC into an expected discovery significance as in Fig. 5 of Ref. [[1]](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html).
* Compare your networks with a boosted decision tree (e.g. `sklearn.ensemble.HistGradientBoostingClassifier` or XGBoost) trained on the same features. BDTs were the standard tool in particle physics at the time.

## <span style="color:Orange">References</span>

__[<span style="color:Red">1</span>]__ P. Sadowski, D. Whiteson, P. Baldi, "Searching for Higgs Boson Decay Modes with Deep Learning," *Advances in Neural Information Processing Systems 27 (NIPS 2014)*. [Abstract](https://papers.nips.cc/paper_files/paper/2014/hash/44f58a78b6dd03bf9ccb526e7c7538d5-Abstract.html), [PDF](https://papers.nips.cc/paper_files/paper/2014/file/44f58a78b6dd03bf9ccb526e7c7538d5-Paper.pdf)

__[<span style="color:Red">2</span>]__ P. Baldi, P. Sadowski, D. Whiteson, "Enhanced Higgs Boson to τ⁺τ⁻ Search with Deep Learning," *Phys. Rev. Lett.* **114**, 111801 (2015). [doi:10.1103/PhysRevLett.114.111801](https://doi.org/10.1103/PhysRevLett.114.111801), [arXiv:1410.3469](https://arxiv.org/abs/1410.3469)

__[<span style="color:Red">3</span>]__ D. Whiteson, Higgs to Tau Tau dataset, UCI MLPhysics data portal. https://mlphysics.ics.uci.edu/data/htautau/

## <span style="color:Orange">Acknowledgements</span>

* Initial version: Mark Neubauer
* Revised: 2026

© Copyright 2026